In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

DATA_DIR = "/Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis"

results_tracker = {
    "run_timestamp": datetime.now().isoformat(),
    "datasets": {},
    "phases": {}
}

def log_metric(phase, name, value):
    if phase not in results_tracker["phases"]:
        results_tracker["phases"][phase] = {}
    results_tracker["phases"][phase][name] = value
    print(f"[{phase}] {name}: {value}")

def load_and_prepare_data(wafer_path, thresholds_path):
    """
    Loads wafer data and builds:
    - continuous matrix
    - binary fail matrix
    - final fail label
    - thresholds dictionary
    """

    print(f"\nLoading: {wafer_path}")

    # ---- Load files ----
    df_raw = pd.read_csv(wafer_path)
    thresholds_df = pd.read_csv(thresholds_path)

    print(f"  Threshold file shape: {thresholds_df.shape}")

    # METADATA COLUMNS TO EXCLUDE
    METADATA_COLUMNS = {
        'Lot', 'Wafer', 'rework_flag', 'Program', 'temperature',
        'subid', 'site', 'die_x', 'die_y', 'device_nr', 'rom_code',
        'hardbin', 'lib_info', 'BinName', 'BinState'
    }

    # Get all columns except the first (die ID) and metadata
    all_test_candidates = df_raw.columns[1:].tolist()
    wafer_test_cols = [col for col in all_test_candidates if col not in METADATA_COLUMNS]

    print(f"  Total columns in file: {len(df_raw.columns)}")
    print(f"  Metadata columns excluded: {len(METADATA_COLUMNS & set(all_test_candidates))}")
    print(f"  Candidate test columns: {len(wafer_test_cols)}")

    # Build threshold dictionary
    thresholds = {}

    for test_col in wafer_test_cols:
        if test_col in thresholds_df.columns:
            lower_val = thresholds_df[test_col].iloc[0]
            upper_val = thresholds_df[test_col].iloc[1]

            try:
                lower_val = float(lower_val)
                upper_val = float(upper_val)

                if lower_val > upper_val:
                    lower_val, upper_val = upper_val, lower_val

                thresholds[test_col] = (lower_val, upper_val)
            except (ValueError, TypeError):
                print(f"  Warning: Could not parse thresholds for {test_col}")
                pass

    # Keep only tests that have thresholds
    test_columns = [col for col in wafer_test_cols if col in thresholds]

    if len(test_columns) == 0:
        print("  ERROR: No matching tests found between wafer data and thresholds file!")
        return None, None, None, [], None, None

    df_cont = df_raw[test_columns].copy()
    die_ids = df_raw.iloc[:, 0]

    nan_mask = df_cont.isna()
    nan_count = nan_mask.sum().sum()

    if nan_count > 0:
        print(f"  Warning: {nan_count} NaN values found (will treat as FAIL)")
        df_cont_filled = df_cont.fillna(0)
    else:
        df_cont_filled = df_cont

    fail_conditions = []

    for col in test_columns:
        lower, upper = thresholds[col]
        if lower == upper:
            fail = df_cont_filled[col] != lower
        else:
            fail = (df_cont_filled[col] < lower) | (df_cont_filled[col] > upper)

        if nan_count > 0:
            fail = fail | nan_mask[col]

        fail_conditions.append(fail)

    if fail_conditions:
        df_bin = pd.concat(fail_conditions, axis=1)
        df_bin.columns = test_columns
        df_bin = df_bin.astype(int)
    else:
        df_bin = pd.DataFrame(index=df_cont.index)

    y_final = df_bin.any(axis=1).astype(int)

    return df_cont, df_bin, y_final, test_columns, die_ids, thresholds

print("=" * 60)
print("PHASE 1: DATA LOADING & VALIDATION")
print("=" * 60)

thresholds_path = os.path.join(DATA_DIR, "thresholds.csv")

print("\n--- Loading Training Data (wafer_801) ---")
train_cont, train_bin, train_y, test_cols, train_ids, thresholds = load_and_prepare_data(
    os.path.join(DATA_DIR, "wafer_801.csv"), thresholds_path
)

if train_cont is not None:
    print("\n--- Loading Validation Data (wafer_806) ---")
    val_cont, val_bin, val_y, _, val_ids, _ = load_and_prepare_data(
        os.path.join(DATA_DIR, "wafer_806.csv"), thresholds_path
    )

    print("\n--- Loading Test Data (wafer_812) ---")
    test_cont, test_bin, test_y, _, test_ids, _ = load_and_prepare_data(
        os.path.join(DATA_DIR, "wafer_812.csv"), thresholds_path
    )

    print(f"\nTotal tests with thresholds: {len(test_cols)}")

    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)

    def dataset_stats(name, y):
        if y is not None:
            total = len(y)
            fails = int(y.sum())
            rate = float(y.mean())
            print(f"{name:10}: {total:8,} samples | {fails:8,} fails | {rate*100:6.3f}%")
            log_metric("Phase 1", f"{name}_samples", total)
            log_metric("Phase 1", f"{name}_fails", fails)
            log_metric("Phase 1", f"{name}_fail_rate", rate)
        else:
            print(f"{name:10}: Data loading failed")

    dataset_stats("Train", train_y)
    dataset_stats("Validation", val_y)
    dataset_stats("Test", test_y)

    log_metric("Phase 1", "num_tests", len(test_cols) if test_cols else 0)

    print(f"\nTotal tests with thresholds: {len(test_cols) if test_cols else 0}")

    if train_bin is not None and len(train_bin.columns) > 0:
        print("\n--- Top 10 tests with highest fail rates (training) ---")
        fail_rates = train_bin.mean().sort_values(ascending=False)
        for test_name, rate in fail_rates.head(10).items():
            if rate > 0:
                print(f"  {test_name[:60]}: {rate*100:.4f}%")

        never_fail_tests = (train_bin.sum() == 0).sum()
        always_fail_tests = (train_bin.sum() == len(train_bin)).sum()

        print(f"\n--- Overall Statistics ---")
        print(f"  Total failing dies: {train_y.sum()}")
        print(f"  Fail rate: {train_y.mean()*100:.4f}%")
        print(f"  Tests with any failures: {(train_bin.sum() > 0).sum()}")
        print(f"  Tests with zero failures: {never_fail_tests}")
        if always_fail_tests > 0:
            print(f"  Tests that always fail: {always_fail_tests}")

    if train_cont is not None:
        print("\n--- Memory Usage ---")
        print(f"  Train continuous data: {train_cont.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        print(f"  Train binary data: {train_bin.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

        if train_bin is not None:
            print(f"\n--- Data Quality ---")
            original_nan_count = train_cont.isna().sum().sum()
            if original_nan_count > 0:
                print(f"  Original NaNs: {original_nan_count}")
                print(f"  (All NaNs treated as failures)")
else:
    print("\nERROR: Failed to load training data. Check file formats.")

# After loading train_cont, train_bin, etc. add:

METADATA_COLUMNS = {
    'Lot', 'Wafer', 'rework_flag', 'Program', 'temperature',
    'subid', 'site', 'die_x', 'die_y', 'device_nr', 'rom_code',
    'hardbin', 'lib_info', 'BinName', 'BinState'
}

# Filter out metadata columns from all dataframes
def filter_metadata(df, metadata_cols):
    if df is not None:
        keep_cols = [col for col in df.columns if col not in metadata_cols]
        return df[keep_cols]
    return df

train_cont = filter_metadata(train_cont, METADATA_COLUMNS)
train_bin = filter_metadata(train_bin, METADATA_COLUMNS)
val_cont = filter_metadata(val_cont, METADATA_COLUMNS)
val_bin = filter_metadata(val_bin, METADATA_COLUMNS)
test_cont = filter_metadata(test_cont, METADATA_COLUMNS)
test_bin = filter_metadata(test_bin, METADATA_COLUMNS)

# Update test_cols
test_cols = [col for col in test_cols if col not in METADATA_COLUMNS]

print(f"\n--- After metadata removal ---")
print(f"Remaining tests: {len(test_cols)}")

print("\n" + "=" * 60)
print("PHASE 2: FIRST-FAIL (UNIQUE FAILURE) ANALYSIS")
print("=" * 60)

def compute_first_failures(df_bin, test_cols):
    first_fail_counts = []
    failed_before = np.zeros(len(df_bin), dtype=bool)

    for i, col in enumerate(test_cols):
        current_fail = df_bin[col].values.astype(bool)
        first_fail = current_fail & (~failed_before)
        count = first_fail.sum()
        first_fail_counts.append(count)
        failed_before = failed_before | current_fail

        if i < 5:
            print(f"  Test {i:3d} ({col[:40]}): first fails = {count}")

    return pd.Series(first_fail_counts, index=test_cols)

train_first_fail = compute_first_failures(train_bin, test_cols)

total_first_fails = train_first_fail.sum()
zero_unique_tests = (train_first_fail == 0).sum()

print("\n--- First-Fail Summary ---")
print(f"Total first-fail detections: {total_first_fails}")
print(f"Tests with ZERO first-fails: {zero_unique_tests} / {len(test_cols)}")

log_metric("Phase 2", "total_first_fails", int(total_first_fails))
log_metric("Phase 2", "zero_unique_tests", int(zero_unique_tests))

print("\n--- Top 10 tests by FIRST-FAIL contribution ---")
top_first_fail = train_first_fail.sort_values(ascending=False).head(10)

for test, count in top_first_fail.items():
    print(f"  {test[:60]}: {count}")

print("\n--- Sample tests with ZERO first-fails (redundancy candidates) ---")
zero_tests = train_first_fail[train_first_fail == 0].index.tolist()

for test in zero_tests[:10]:
    print(f"  {test[:60]}")

print("\n--- Sanity Check ---")
print(f"Train total fails: {train_y.sum()}")
print(f"Sum of first-fails: {total_first_fails}")

if total_first_fails != train_y.sum():
    print("WARNING: mismatch in fail accounting!")
else:
    print("First-fail accounting is consistent")

PHASE 1: DATA LOADING & VALIDATION

--- Loading Training Data (wafer_801) ---

Loading: /Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis/wafer_801.csv
  Threshold file shape: (2, 588)
  Total columns in file: 588
  Metadata columns excluded: 15
  Candidate test columns: 572

--- Loading Validation Data (wafer_806) ---

Loading: /Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis/wafer_806.csv
  Threshold file shape: (2, 588)
  Total columns in file: 588
  Metadata columns excluded: 15
  Candidate test columns: 572

--- Loading Test Data (wafer_812) ---

Loading: /Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis/wafer_812.csv
  Threshold file shape: (2, 588)
  Total columns in file: 588
  Metadata columns excluded: 15
  Candidate test columns: 572

Total tests with thresholds: 572

DATASET SUMMARY
Train     :   26,149 samples |      127 fails |  0.486%
[Phase 1] Train_samples: 26149
[Phase 1] Train_fails: 127
[Phase 1] Train_fail_rate: 0.004856782286129489
Validat

In [4]:
# =========================
# PHASE 10: CONTINUOUS MARGIN ANALYSIS
# =========================

print("\n" + "=" * 60)
print("PHASE 10: CONTINUOUS MARGIN ANALYSIS")
print("=" * 60)


# =========================
# BUILD NORMALIZED MARGIN MATRIX
# =========================
def compute_margin_matrix(df_cont, thresholds):
    """
    Efficient margin computation (no fragmentation)
    """
    margin_data = {}

    for col in df_cont.columns:
        if col not in thresholds:
            continue

        lower, upper = thresholds[col]
        values = df_cont[col]

        dist_lower = values - lower
        dist_upper = upper - values
        margin = np.minimum(dist_lower, dist_upper)

        range_width = (upper - lower)
        if range_width > 0:
            margin = margin / range_width

        margin_data[col] = margin

    # build dataframe in one shot (FAST)
    df_margin = pd.DataFrame(margin_data)

    return df_margin


# ---- Compute ----
train_margin = compute_margin_matrix(train_cont, thresholds)

print(f"\nMargin matrix shape: {train_margin.shape}")


# =========================
# SANITY CHECKS
# =========================

print("\n--- Margin Statistics ---")

print("Global stats:")
print(train_margin.describe().loc[['min', 'mean', 'max']])

# Check consistency with binary fails
print("\n--- Consistency Check (margin vs fail) ---")

fail_from_margin = (train_margin < 0).any(axis=1).astype(int)

mismatch = (fail_from_margin != train_y).sum()

print(f"Mismatches with binary labels: {mismatch}")

if mismatch == 0:
    print("✅ Margin correctly reproduces failure labels")
else:
    print("⚠️ Some mismatch detected")


# =========================
# IDENTIFY FAMILIES
# =========================

def extract_family_name(test_name):
    """
    Extract family prefix (before first ':')
    """
    if ":" in test_name:
        return test_name.split(":")[0]
    return test_name


# Build family mapping
family_map = {}
for col in train_margin.columns:
    fam = extract_family_name(col)
    family_map.setdefault(fam, []).append(col)

print(f"\nTotal families detected: {len(family_map)}")


# =========================
# PICK A GOOD FAMILY (AUTO)
# =========================

# choose family with multiple tests and variability
candidate_families = [
    f for f, tests in family_map.items() if len(tests) >= 4
]

selected_family = candidate_families[0] if candidate_families else None

print(f"\nSelected family: {selected_family}")


# =========================
# ANALYZE FAMILY STRUCTURE
# =========================

if selected_family is not None:
    family_tests = family_map[selected_family]

    print(f"\nFamily size: {len(family_tests)}")

    # ---- Correlation ----
    print("\n--- Correlation within family ---")
    corr_matrix = train_margin[family_tests].corr()

    print(corr_matrix.round(2))

    # =========================
    # SAMPLE CURVES (first few dies)
    # =========================

    print("\n--- Sample margin curves (first 5 dies) ---")

    sample_dies = train_margin.iloc[:5][family_tests]

    for i, row in sample_dies.iterrows():
        print(f"\nDie {i}:")
        for test, val in row.items():
            print(f"  {test[:50]:50} → {val:.4f}")


# =========================
# OPTIONAL: SORT FAMILY TESTS (if numeric parameter exists)
# =========================

import re

def extract_numeric_param(test_name):
    """
    Extract numeric value (e.g., voltage) from test name
    """
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", test_name)
    return float(nums[-1]) if nums else np.nan


if selected_family is not None:
    sorted_tests = sorted(
        family_tests,
        key=lambda x: extract_numeric_param(x)
    )

    print("\n--- Sorted family tests (by parameter) ---")
    for t in sorted_tests:
        print(f"  {t}")


# =========================
# LOG
# =========================
# =========================

log_metric("Phase 10", "num_families", len(family_map))
log_metric("Phase 10", "selected_family_size", len(family_tests) if selected_family else 0)


PHASE 10: CONTINUOUS MARGIN ANALYSIS

Margin matrix shape: (26149, 572)

--- Margin Statistics ---
Global stats:
      tb_cont_all:CONT_VCC:-5.000 mA  tb_cont_all:CONT_N:-5.000 mA  \
min                        -0.399375                      0.402000   
mean                        0.384165                      0.450581   
max                         0.451875                      0.458000   

      tb_cont_all:CONT_N:-5.000 mA (2)  tb_cont_all:CONT_N:-5.000 mA (3)  \
min                           0.360000                          -0.94000   
mean                          0.390453                           0.37721   
max                           0.400000                           0.38400   

      tb_cont_all:CONT_N:-5.000 mA (4)  tb_cont_all:CONT_N:-5.000 mA (5)  \
min                          -0.922000                          0.362000   
mean                          0.377215                          0.378384   
max                           0.432000                          0.386000

In [5]:
# Print as a list
print(list(train_margin.columns))

['tb_cont_all:CONT_VCC:-5.000 mA', 'tb_cont_all:CONT_N:-5.000 mA', 'tb_cont_all:CONT_N:-5.000 mA (2)', 'tb_cont_all:CONT_N:-5.000 mA (3)', 'tb_cont_all:CONT_N:-5.000 mA (4)', 'tb_cont_all:CONT_N:-5.000 mA (5)', 'tb_cont_all:CONT_N:-5.000 mA (6)', 'tb_cont_all:CONT_N:-5.000 mA (7)', 'tb_cont_all:CONT_N:-5.000 mA (8)', 'tb_cont_all:CONT_N:-5.000 mA (9)', 'tb_cont_all:CONT_N:-5.000 mA (10)', 'tb_cont_all:CONT_N:-5.000 mA (11)', 'tb_cont_all:CONT_N:-5.000 mA (12)', 'tb_cont_all:CONT_N:-5.000 mA (13)', 'tb_cont_all:CONT_N:-5.000 mA (14)', 'tb_cont_all:CONT_N:-5.000 mA (15)', 'tb_cont_all:CONT_N:-5.000 mA (16)', 'tb_cont_all:CONT_N:-5.000 mA (17)', 'tb_cont_all:CONT_N:-5.000 mA (18)', 'tb_cont_all:CONT_N:-5.000 mA (19)', 'tb_cont_all:CONT_N:-5.000 mA (20)', 'tb_cont_all:CONT_N:-5.000 mA (21)', 'tb_cont_all:CONT_N:-5.000 mA (22)', 'tf_ICC_SCL:VCC_ICC_SCL:1.650 V', 'tf_ICC_SCL:VCC_ICC_SCL:2.300 V', 'tf_ICC_SCL:VCC_ICC_SCL:3.600 V', 'tf_ICC_SCL:VCC_ICC_SCL:5.500 V', 'tf_ICC_Op:ICC_Op:1.650 V', 

In [12]:
# Quick summary
total_values = train_margin.size
failing = (train_margin < 0).sum().sum()
zero = (train_margin == 0).sum().sum()
passing = total_values - failing - zero

print(f"Total margin values: {total_values:,}")
print(f"  FAILING (< 0):    {failing:>10,} ({failing/total_values*100:.2f}%)")
print(f"  ZERO (== 0):      {zero:>10,} ({zero/total_values*100:.2f}%)")
print(f"  PASSING (> 0):    {passing:>10,} ({passing/total_values*100:.2f}%)")

Total margin values: 14,957,228
  FAILING (< 0):         3,449 (0.02%)
  ZERO (== 0):               5 (0.00%)
  PASSING (> 0):    14,953,774 (99.98%)


In [23]:
import re

def parse_column(col_name):
    """
    Extract structure from column name
    """

    # Split main parts
    parts = col_name.split(":")

    if len(parts) < 2:
        return None

    family = parts[0]
    test_part = parts[1]

    # Try to extract pin index (e.g., "(3)")
    pin_match = re.search(r"\((\d+)\)", col_name)
    pin = int(pin_match.group(1)) if pin_match else None

    # Extract voltage / condition from test_part
    # Example: IOL_1v65 → test=IOL, voltage=1v65
    tokens = test_part.split("_")

    test = tokens[0]
    condition_tokens = tokens[1:]

    # Try to detect voltage/current
    voltage = None
    current = None

    for t in condition_tokens:
        if "v" in t.lower():
            voltage = t
        elif "m" in t.lower():
            current = t

    return {
        "family": family,
        "test": test,
        "condition": test_part,
        "voltage": voltage,
        "current": current,
        "pin": pin
    }

In [24]:
from collections import defaultdict

def build_groups(df):
    groups = defaultdict(list)
    meta = {}

    for col in df.columns:
        parsed = parse_column(col)
        if parsed is None:
            continue

        # grouping key (NO pin)
        key = (
            parsed["family"],
            parsed["test"],
            parsed["condition"]
        )

        groups[key].append(col)
        meta[key] = parsed

    return groups, meta

In [25]:
import pandas as pd
import numpy as np

def aggregate_groups(df_margin, groups):
    feature_data = {}

    for key, cols in groups.items():
        family, test, condition = key

        group_name = f"{family}_{condition}"

        values = df_margin[cols]

        if len(cols) > 1:
            # multi-pin → aggregate
            feature_data[group_name + "_min"] = values.min(axis=1)
            feature_data[group_name + "_mean"] = values.mean(axis=1)
            feature_data[group_name + "_std"] = values.std(axis=1)
        else:
            # single measurement → keep as is
            feature_data[group_name] = values.iloc[:, 0]

    df_features = pd.DataFrame(feature_data)

    return df_features

In [26]:
def build_feature_matrix(df_margin):
    print("Parsing and grouping columns...")

    groups, meta = build_groups(df_margin)

    print(f"Total groups: {len(groups)}")

    print("Aggregating features...")

    df_features = aggregate_groups(df_margin, groups)

    print(f"Final feature shape: {df_features.shape}")

    return df_features, groups, meta

In [28]:
df_features, groups, meta = build_feature_matrix(train_margin)

Parsing and grouping columns...
Total groups: 40
Aggregating features...
Final feature shape: (26149, 118)


In [33]:
# check one group
example_key = list(groups.keys())[4]
print(example_key)
print(groups[example_key])

('tf_ICC_stand', 'VCC', 'VCC_ICC_standby')
['tf_ICC_stand:VCC_ICC_standby:1.650 V', 'tf_ICC_stand:VCC_ICC_standby:2.300 V', 'tf_ICC_stand:VCC_ICC_standby:3.600 V', 'tf_ICC_stand:VCC_ICC_standby:5.500 V']


In [34]:
from collections import defaultdict

def build_hierarchy(groups):
    tree = defaultdict(lambda: defaultdict(dict))

    for key, cols in groups.items():
        family, test, condition = key

        tree[family][test][condition] = len(cols)

    return tree

In [36]:
def print_tree(tree):
    for family, tests in tree.items():
        print(f"\n{family}")
        for test, conditions in tests.items():
            print(f"  ├── {test}")
            for cond, count in conditions.items():
                print(f"      ├── {cond}  ({count} columns)")

In [37]:
tree = build_hierarchy(groups)
print_tree(tree)


tb_cont_all
  ├── CONT
      ├── CONT_VCC  (1 columns)
      ├── CONT_N  (22 columns)

tf_ICC_SCL
  ├── VCC
      ├── VCC_ICC_SCL  (4 columns)

tf_ICC_Op
  ├── ICC
      ├── ICC_Op  (4 columns)

tf_ICC_stand
  ├── VCC
      ├── VCC_ICC_standby  (4 columns)

tf_ICC_PullU
  ├── ICC
      ├── ICC_PullUp_VSS  (2 columns)

tf_ICCQ_SCL
  ├── ICCQ
      ├── ICCQ_SCL  (2 columns)

tf_ICCQ_Ppor
  ├── ICCQ
      ├── ICCQ_Pport  (2 columns)

tf_dIDDq1
  ├── IDDq1
      ├── IDDq1  (19 columns)

tb_iiL_Pport
  ├── iiL
      ├── iiL_Pport_1v65_  (16 columns)
      ├── iiL_Pport_5v5_v  (16 columns)

tb_iinL
  ├── iinL
      ├── iinL_1v65_vi_0  (5 columns)
      ├── iinL_1v65_vi_1v  (5 columns)
      ├── iinL_5v5_vi_0  (5 columns)
      ├── iinL_5v5_vi_5v5  (5 columns)

tb_iinH
  ├── iinH
      ├── iinH_1v65_vi_0  (5 columns)
      ├── iinH_1v65_vi_1v  (5 columns)
      ├── iinH_5v5_vi_0  (5 columns)
      ├── iinH_5v5_vi_5v5  (5 columns)

tb_iiH_Pport
  ├── ii
      ├── ii_Pport_1v65_v  (16 columns)

In [38]:
# ============================================
# SANITY CHECKS (Run on train_margin BEFORE aggregation)
# ============================================

# Check 1: Column counts from your tree
tree_checks = {
    'CONT_N': 22,
    'iiL_Pport_1v65_': 16,
    'IOL_Ppprt_1v65': 32,  # 16 pins × 2 loads
    'VOH_8m': 64,
    'I2C_NACK': 16,
}

print("=== SANITY CHECKS ===\n")

for pattern, expected in tree_checks.items():
    matches = [c for c in train_margin.columns if pattern in c]
    print(f"{pattern}: found {len(matches)} columns (expected {expected})")

# Check 2: Pin consistency (pick first multi-pin group)
test_cols = [c for c in train_margin.columns if 'I2C_NACK' in c][:16]
if test_cols:
    values = train_margin[test_cols].iloc[0]  # first device
    print(f"\nPin consistency (I2C_NACK):")
    print(f"  Min: {values.min():.3f}, Max: {values.max():.3f}, Std: {values.std():.3f}")

# Check 3: Missing data
missing = train_margin.isna().sum().sum() / (train_margin.shape[0] * train_margin.shape[1])
print(f"\nMissing data: {missing:.2%}")

# Check 4: Pin correlation (are replicas actually correlated?)
if len(test_cols) >= 2:
    corr = train_margin[test_cols[0]].corr(train_margin[test_cols[1]])
    print(f"Pin-to-pin correlation: {corr:.3f} (should be >0.8)")

print("\n" + "="*40)

=== SANITY CHECKS ===

CONT_N: found 22 columns (expected 22)
iiL_Pport_1v65_: found 16 columns (expected 16)
IOL_Ppprt_1v65: found 32 columns (expected 32)
VOH_8m: found 64 columns (expected 64)
I2C_NACK: found 16 columns (expected 16)

Pin consistency (I2C_NACK):
  Min: 0.302, Max: 0.314, Std: 0.003

Missing data: 0.00%
Pin-to-pin correlation: -0.041 (should be >0.8)



In [40]:
# Check I2C_NACK pins on ORIGINAL data (df_raw from load_and_prepare_data)
# But note: df_raw is not saved - we need to reload or access from within the function

# Since df_raw is not stored globally, let's access from the original dataframes
# train_cont contains the continuous values (original measurements, NOT margins yet)

print("\n" + "=" * 60)
print("DIAGNOSTIC: I2C_NACK PIN ANALYSIS (ORIGINAL DATA)")
print("=" * 60)

# Find all I2C_NACK columns in train_cont (original continuous data)
i2c_cols_raw = [c for c in train_cont.columns if 'I2C_NACK' in c][:16]

print(f"Found {len(i2c_cols_raw)} I2C_NACK columns")
print(f"First 5 columns: {i2c_cols_raw[:5]}\n")

# Take first device (die 0)
first_device_raw = train_cont.iloc[0]

print("=== FIRST DIE (original raw values) ===")
values_raw = []
for col in i2c_cols_raw[:5]:  # Show first 5 only
    val = first_device_raw[col]
    values_raw.append(val)
    print(f"{col[:55]:55} → {val}")

# Full stats on first 16
values_raw_full = [first_device_raw[col] for col in i2c_cols_raw]
print(f"\nStats across all 16 pins on first die:")
print(f"  Min: {min(values_raw_full):.6f}")
print(f"  Max: {max(values_raw_full):.6f}")
print(f"  Mean: {np.mean(values_raw_full):.6f}")
print(f"  Std: {np.std(values_raw_full):.6f}")
print(f"  Range: {max(values_raw_full) - min(values_raw_full):.6f}")

if np.std(values_raw_full) < 0.01:
    print("  ✅ Pins are CONSISTENT on same die (std very small)")
else:
    print("  ⚠️ Pins vary significantly on same die")

# Correlation across ALL devices (raw data)
print("\n=== CORRELATION ACROSS ALL DIES (original data) ===")
if len(i2c_cols_raw) >= 2:
    corr_raw = train_cont[i2c_cols_raw[0]].corr(train_cont[i2c_cols_raw[1]])
    print(f"Pin1 vs Pin2 correlation: {corr_raw:.4f}")

    if len(i2c_cols_raw) >= 4:
        corr_raw_2 = train_cont[i2c_cols_raw[2]].corr(train_cont[i2c_cols_raw[3]])
        print(f"Pin3 vs Pin4 correlation: {corr_raw:.4f}")

    # What correlation SHOULD be for replicas
    if corr_raw > 0.95:
        print("  ✅ Excellent correlation - pins ARE replicas")
    elif corr_raw > 0.8:
        print("  ⚠️ Moderate correlation - may be replicas with noise")
    else:
        print("  ❌ Poor correlation - these are NOT replica pins")

# Check distribution across all dies for first few pins
print("\n=== DISTRIBUTION (all dies) ===")
for col in i2c_cols_raw[:3]:
    print(f"{col[:50]:50} mean={train_cont[col].mean():.4f}, std={train_cont[col].std():.4f}")


DIAGNOSTIC: I2C_NACK PIN ANALYSIS (ORIGINAL DATA)
Found 16 I2C_NACK columns
First 5 columns: ['NCA_pullup:I2C_NACK:300.000 mV', 'NCA_pullup:I2C_NACK:300.000 mV (2)', 'NCA_pullup:I2C_NACK:300.000 mV (3)', 'NCA_pullup:I2C_NACK:300.000 mV (4)', 'NCA_pullup:I2C_NACK:300.000 mV (5)']

=== FIRST DIE (original raw values) ===
NCA_pullup:I2C_NACK:300.000 mV                          → -15.008000373840298
NCA_pullup:I2C_NACK:300.000 mV (2)                      → -15.1840000152588
NCA_pullup:I2C_NACK:300.000 mV (3)                      → -15.0559997558594
NCA_pullup:I2C_NACK:300.000 mV (4)                      → -15.1520004272461
NCA_pullup:I2C_NACK:300.000 mV (5)                      → -15.1199998855591

Stats across all 16 pins on first die:
  Min: -15.840000
  Max: -14.880000
  Mean: -15.205000
  Std: 0.239180
  Range: 0.960000
  ⚠️ Pins vary significantly on same die

=== CORRELATION ACROSS ALL DIES (original data) ===
Pin1 vs Pin2 correlation: -0.0396
Pin3 vs Pin4 correlation: -0.0396
  ❌ P